# Cortex AI Functions — Cost Governance Toolkit

Monitor, alert on, and control Cortex AI Function spend using `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY`.

**Sections:**
- **A — Usage Monitoring** — Daily, weekly, and per-user credit consumption
- **B — Account-Level Spending Alert** — Monthly threshold alert with email notification
- **C — Per-User Spending Limits** — Role-based access control with individual credit budgets
- **D — Runaway Query Detection** — Automatic cancellation of expensive in-flight queries
- **E — Verification** — Status checks for all governance objects

**Prerequisites:**
- `ACCOUNTADMIN` role (for notification integration, alert, and role creation)
- `GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE <your_role>`

**Data source:** [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history) — up to 60 min latency, 1-hour aggregation windows.

> Source: [Managing Cortex AI Function costs with Account Usage](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)

In [ ]:
-- Setup: use the tool warehouse and schema
USE WAREHOUSE SFE_AI_SPEND_CONTROLS_WH;
USE SCHEMA SNOWFLAKE_EXAMPLE.AI_SPEND_CONTROLS;

---
## A — Usage Monitoring
Understand baseline spending patterns before implementing controls.

### A1. Daily Credit Consumption by Function & Model (30 days)
Identify daily spending spikes and which functions/models drive costs.

In [ ]:
SELECT
    DATE_TRUNC('day', START_TIME)   AS usage_date,
    FUNCTION_NAME,
    MODEL_NAME,
    SUM(CREDITS)                    AS total_credits,
    COUNT(DISTINCT QUERY_ID)        AS query_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY usage_date DESC, total_credits DESC;

### A2. Monthly Credit Consumption by User
Top consumers with email and default role for follow-up.

In [ ]:
SELECT
    DATE_TRUNC('month', h.START_TIME)  AS usage_month,
    u.NAME                             AS user_name,
    u.EMAIL,
    u.DEFAULT_ROLE,
    SUM(h.CREDITS)                     AS total_credits,
    COUNT(DISTINCT h.QUERY_ID)         AS query_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
    ON h.USER_ID = u.USER_ID
WHERE h.START_TIME >= DATEADD('month', -3, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3, 4
ORDER BY usage_month DESC, total_credits DESC;

### A3. Credit Consumption by Model (30 days)
Which models cost the most? Use this to guide model selection policy.

In [ ]:
SELECT
    MODEL_NAME,
    FUNCTION_NAME,
    SUM(CREDITS)                    AS total_credits,
    COUNT(DISTINCT QUERY_ID)        AS query_count,
    ROUND(AVG(CREDITS), 6)          AS avg_credits_per_query
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
  AND MODEL_NAME IS NOT NULL
  AND MODEL_NAME != ''
GROUP BY 1, 2
ORDER BY total_credits DESC;

---
## B — Account-Level Monthly Spending Alert

Set up an automated alert that monitors total monthly AI Function credit consumption. When spending exceeds a threshold, an email notification is sent.

**What gets created:**
- Notification integration (`SFE_AI_COST_ALERTS`)
- Alert state table (`AI_FUNCTIONS_ALERT_STATE`)
- Stored procedure (`SEND_MONTHLY_SPEND_ALERT`)
- Hourly alert (`AI_FUNCTIONS_MONTHLY_SPEND_ALERT`)

**⚠️ Edit the email addresses and credit threshold before running.**

In [ ]:
-- B1. Create notification integration (requires ACCOUNTADMIN)
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE NOTIFICATION INTEGRATION SFE_AI_COST_ALERTS
    TYPE                = EMAIL
    ENABLED             = TRUE
    ALLOWED_RECIPIENTS  = ('admin@company.com', 'finops@company.com')  -- ← EDIT
    COMMENT             = 'TOOL: Cortex AI cost alerts (Expires: 2026-05-07)';

In [ ]:
-- B2. Alert state table (prevents duplicate alerts within a month)
USE ROLE SYSADMIN;
USE SCHEMA SNOWFLAKE_EXAMPLE.AI_SPEND_CONTROLS;

CREATE TABLE IF NOT EXISTS AI_FUNCTIONS_ALERT_STATE (
    ALERT_NAME       VARCHAR NOT NULL,
    ALERT_MONTH      DATE NOT NULL,
    SENT_AT          TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    CREDITS_AT_ALERT NUMBER(38,6),
    PRIMARY KEY (ALERT_NAME, ALERT_MONTH)
)
COMMENT = 'TOOL: Monthly spend alert dedup state (Expires: 2026-05-07)';

In [ ]:
-- B3. Stored procedure: check threshold → record state → send email
CREATE OR REPLACE PROCEDURE SEND_MONTHLY_SPEND_ALERT(P_THRESHOLD FLOAT)
RETURNS VARCHAR
LANGUAGE JAVASCRIPT
EXECUTE AS CALLER
AS
$$
    var check_sent = snowflake.execute({
        sqlText: `SELECT COUNT(*) AS cnt FROM AI_FUNCTIONS_ALERT_STATE
                  WHERE ALERT_NAME = 'monthly_spend'
                    AND ALERT_MONTH = DATE_TRUNC('month', CURRENT_DATE())`
    });
    check_sent.next();
    if (check_sent.getColumnValue(1) > 0) {
        return 'Alert already sent for this month';
    }

    var spend_result = snowflake.execute({
        sqlText: `SELECT COALESCE(SUM(CREDITS), 0) AS total
                  FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
                  WHERE START_TIME >= DATE_TRUNC('month', CURRENT_TIMESTAMP())`
    });
    spend_result.next();
    var v_credits = spend_result.getColumnValue(1);

    if (v_credits <= P_THRESHOLD) {
        return 'Threshold not exceeded. Current: ' + v_credits + ' / ' + P_THRESHOLD;
    }

    snowflake.execute({
        sqlText: `INSERT INTO AI_FUNCTIONS_ALERT_STATE (ALERT_NAME, ALERT_MONTH, CREDITS_AT_ALERT)
                  VALUES ('monthly_spend', DATE_TRUNC('month', CURRENT_DATE()), ?)`,
        binds: [v_credits]
    });

    snowflake.execute({
        sqlText: `CALL SYSTEM$SEND_EMAIL(
            'SFE_AI_COST_ALERTS',
            'admin@company.com',
            'AI Functions Monthly Spend Alert',
            'Monthly AI Function credit consumption has exceeded the threshold.\n\n' ||
            'Current spend: ' || ${v_credits}::VARCHAR || ' credits\n' ||
            'Threshold: ' || ${P_THRESHOLD}::VARCHAR || ' credits\n\n' ||
            'Please review usage accordingly.'
        )`
    });

    return 'Alert sent. Credits: ' + v_credits;
$$
COMMENT = 'TOOL: Check and send monthly AI spend alert (Expires: 2026-05-07)';

In [ ]:
-- B4. Hourly alert — fires when monthly spend exceeds threshold
-- ⚠️ Adjust the 1000 credit limit in BOTH places below
CREATE OR REPLACE ALERT AI_FUNCTIONS_MONTHLY_SPEND_ALERT
    WAREHOUSE = SFE_AI_SPEND_CONTROLS_WH
    SCHEDULE  = 'USING CRON 0 * * * * UTC'
    IF (EXISTS (
        SELECT 1
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
        WHERE START_TIME >= DATE_TRUNC('month', CURRENT_TIMESTAMP())
        HAVING SUM(CREDITS) > 1000
    ))
    THEN
        CALL SEND_MONTHLY_SPEND_ALERT(1000);

ALTER ALERT AI_FUNCTIONS_MONTHLY_SPEND_ALERT RESUME;

SELECT 'Account-level monthly spending alert is now active.' AS status;

---
## C — Per-User Monthly Spending Limits

Implement individual credit budgets per user. When a user exceeds their budget, an hourly task revokes their `AI_FUNCTIONS_USER_ROLE`. A monthly task restores access on the 1st.

**⚠️ Important:** This pattern requires revoking `SNOWFLAKE.CORTEX_USER` from `PUBLIC`:
```sql
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;
```
Ensure all users who need AI Function access are granted `AI_FUNCTIONS_USER_ROLE`.

In [ ]:
-- C1. Create role + access control table
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS AI_FUNCTIONS_USER_ROLE
    COMMENT = 'TOOL: Controls access to Cortex AI Functions (Expires: 2026-05-07)';

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE AI_FUNCTIONS_USER_ROLE;
GRANT USAGE ON WAREHOUSE SFE_AI_SPEND_CONTROLS_WH TO ROLE AI_FUNCTIONS_USER_ROLE;

USE ROLE SYSADMIN;
USE SCHEMA SNOWFLAKE_EXAMPLE.AI_SPEND_CONTROLS;

CREATE TABLE IF NOT EXISTS AI_FUNCTIONS_ACCESS_CONTROL (
    USER_NAME            VARCHAR NOT NULL,
    USER_ID              NUMBER,
    GRANTED_AT           TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    MONTHLY_CREDIT_LIMIT NUMBER(38,6) DEFAULT 100,
    IS_ACTIVE            BOOLEAN DEFAULT TRUE,
    REVOKED_AT           TIMESTAMP_LTZ,
    REVOCATION_REASON    VARCHAR,
    PRIMARY KEY (USER_NAME)
)
COMMENT = 'TOOL: Per-user AI Functions spending limits (Expires: 2026-05-07)';

In [ ]:
-- C2. Procedure to grant access and register a user's spending limit
CREATE OR REPLACE PROCEDURE GRANT_AI_FUNCTIONS_ACCESS(
    P_USER_NAME VARCHAR,
    P_MONTHLY_LIMIT NUMBER(38,6) DEFAULT 100
)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    v_user_id NUMBER;
BEGIN
    SELECT USER_ID INTO :v_user_id
    FROM SNOWFLAKE.ACCOUNT_USAGE.USERS
    WHERE NAME = :P_USER_NAME
    LIMIT 1;

    EXECUTE IMMEDIATE 'GRANT ROLE AI_FUNCTIONS_USER_ROLE TO USER ' || P_USER_NAME;

    MERGE INTO AI_FUNCTIONS_ACCESS_CONTROL tgt
    USING (SELECT :P_USER_NAME AS USER_NAME) src
    ON tgt.USER_NAME = src.USER_NAME
    WHEN MATCHED THEN
        UPDATE SET
            USER_ID = :v_user_id,
            IS_ACTIVE = TRUE,
            MONTHLY_CREDIT_LIMIT = :P_MONTHLY_LIMIT,
            GRANTED_AT = CURRENT_TIMESTAMP(),
            REVOKED_AT = NULL,
            REVOCATION_REASON = NULL
    WHEN NOT MATCHED THEN
        INSERT (USER_NAME, USER_ID, MONTHLY_CREDIT_LIMIT, IS_ACTIVE)
        VALUES (:P_USER_NAME, :v_user_id, :P_MONTHLY_LIMIT, TRUE);

    RETURN 'Access granted to ' || P_USER_NAME || ' with monthly limit of ' || P_MONTHLY_LIMIT || ' credits';
END;
$$
COMMENT = 'TOOL: Grant AI Functions access with spending limit (Expires: 2026-05-07)';

In [ ]:
-- C3. Example: grant access to users (edit names and limits)
-- CALL GRANT_AI_FUNCTIONS_ACCESS('ALICE', 1000);
-- CALL GRANT_AI_FUNCTIONS_ACCESS('BOB', 2000);

SELECT 'Edit and uncomment the CALL statements above to grant access.' AS instructions;

In [ ]:
-- C4. Monthly task: restore access on the 1st of each month
CREATE OR REPLACE PROCEDURE GRANT_ALL_ENTITLED_USERS()
RETURNS TABLE (USER_NAME VARCHAR, CREDIT_LIMIT NUMBER, ACTION VARCHAR)
LANGUAGE SQL
AS
$$
DECLARE
    result RESULTSET;
BEGIN
    result := (
        SELECT
            USER_NAME,
            MONTHLY_CREDIT_LIMIT AS CREDIT_LIMIT,
            'GRANTED' AS ACTION
        FROM AI_FUNCTIONS_ACCESS_CONTROL
    );

    FOR rec IN result DO
        CALL GRANT_AI_FUNCTIONS_ACCESS(rec.USER_NAME, rec.CREDIT_LIMIT);
    END FOR;

    RETURN TABLE(result);
END;
$$
COMMENT = 'TOOL: Re-grant AI Functions access to all entitled users (Expires: 2026-05-07)';

CREATE OR REPLACE TASK MONTHLY_AI_FUNCTIONS_ACCESS_REFRESH
    WAREHOUSE = SFE_AI_SPEND_CONTROLS_WH
    SCHEDULE  = 'USING CRON 0 0 1 * * UTC'
AS
    CALL GRANT_ALL_ENTITLED_USERS();

ALTER TASK MONTHLY_AI_FUNCTIONS_ACCESS_REFRESH RESUME;

SELECT 'Monthly access refresh task is active — runs on the 1st of each month.' AS status;

In [ ]:
-- C5. Hourly task: revoke access for users who exceed their monthly limit
CREATE OR REPLACE PROCEDURE ENFORCE_AI_FUNCTIONS_LIMITS()
RETURNS TABLE (USER_NAME VARCHAR, CREDITS_USED NUMBER, CREDIT_LIMIT NUMBER, ACTION VARCHAR)
LANGUAGE SQL
AS
$$
DECLARE
    result RESULTSET;
BEGIN
    result := (
        SELECT
            ac.USER_NAME,
            COALESCE(SUM(h.CREDITS), 0) AS CREDITS_USED,
            ac.MONTHLY_CREDIT_LIMIT     AS CREDIT_LIMIT,
            'REVOKED'                   AS ACTION
        FROM AI_FUNCTIONS_ACCESS_CONTROL ac
        LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
            ON ac.USER_ID = h.USER_ID
           AND h.START_TIME >= DATE_TRUNC('month', CURRENT_TIMESTAMP())
        WHERE ac.IS_ACTIVE = TRUE
        GROUP BY ac.USER_NAME, ac.MONTHLY_CREDIT_LIMIT
        HAVING COALESCE(SUM(h.CREDITS), 0) > ac.MONTHLY_CREDIT_LIMIT
    );

    FOR rec IN result DO
        EXECUTE IMMEDIATE 'REVOKE ROLE AI_FUNCTIONS_USER_ROLE FROM USER ' || rec.USER_NAME;
        UPDATE AI_FUNCTIONS_ACCESS_CONTROL
            SET IS_ACTIVE = FALSE,
                REVOKED_AT = CURRENT_TIMESTAMP(),
                REVOCATION_REASON = 'Monthly limit exceeded: ' || rec.CREDITS_USED || ' / ' || rec.CREDIT_LIMIT
            WHERE USER_NAME = rec.USER_NAME;
    END FOR;

    RETURN TABLE(result);
END;
$$
COMMENT = 'TOOL: Revoke AI Functions access for over-limit users (Expires: 2026-05-07)';

CREATE OR REPLACE TASK ENFORCE_AI_FUNCTIONS_LIMITS_TASK
    WAREHOUSE = SFE_AI_SPEND_CONTROLS_WH
    SCHEDULE  = 'USING CRON 0 * * * * UTC'
AS
    CALL ENFORCE_AI_FUNCTIONS_LIMITS();

ALTER TASK ENFORCE_AI_FUNCTIONS_LIMITS_TASK RESUME;

SELECT 'Per-user spending limit enforcement task is active (hourly).' AS status;

---
## D — Runaway Query Detection & Cancellation

Automatically detect and cancel AI Function queries that exceed a credit threshold while still running. An email alert is sent with full query details.

**Note:** When a query is cancelled, credits already consumed are still charged. Cancellation prevents further accumulation.

In [ ]:
-- D1. Procedure: find expensive in-flight queries, cancel, and alert
CREATE OR REPLACE PROCEDURE MONITOR_AND_CANCEL_RUNAWAY_QUERIES(
    P_CREDIT_THRESHOLD NUMBER DEFAULT 50
)
RETURNS TABLE (
    QUERY_ID VARCHAR,
    USER_NAME VARCHAR,
    FUNCTION_NAME VARCHAR,
    MODEL_NAME VARCHAR,
    CREDITS NUMBER,
    START_TIME TIMESTAMP_LTZ,
    ACTION VARCHAR
)
LANGUAGE SQL
AS
$$
DECLARE
    result RESULTSET;
BEGIN
    result := (
        SELECT
            h.QUERY_ID,
            u.NAME                  AS USER_NAME,
            h.FUNCTION_NAME,
            h.MODEL_NAME,
            h.CREDITS,
            h.START_TIME,
            h.ROLE_NAMES,
            h.QUERY_TAG,
            h.WAREHOUSE_ID,
            'CANCELLED'             AS ACTION
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
        LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
            ON h.USER_ID = u.USER_ID
        WHERE h.START_TIME >= DATEADD('hour', -48, CURRENT_TIMESTAMP())
          AND h.CREDITS > :P_CREDIT_THRESHOLD
          AND h.IS_COMPLETED = FALSE
    );

    FOR rec IN result DO
        BEGIN
            EXECUTE IMMEDIATE 'SELECT SYSTEM$CANCEL_QUERY(''' || rec.QUERY_ID || ''')';
        EXCEPTION
            WHEN OTHER THEN
                NULL;
        END;

        CALL SYSTEM$SEND_EMAIL(
            'SFE_AI_COST_ALERTS',
            'admin@company.com',
            'Runaway AI Query Cancelled - ' || rec.QUERY_ID,
            'A runaway AI Function query has been cancelled due to excessive cost.\n\n' ||
            'Query Details:\n' ||
            '- Query ID: ' || rec.QUERY_ID || '\n' ||
            '- User: ' || COALESCE(rec.USER_NAME, 'Unknown') || '\n' ||
            '- Function: ' || rec.FUNCTION_NAME || '\n' ||
            '- Model: ' || rec.MODEL_NAME || '\n' ||
            '- Credits Used: ' || rec.CREDITS::VARCHAR || '\n' ||
            '- Threshold: ' || :P_CREDIT_THRESHOLD::VARCHAR || '\n' ||
            '- Start Time: ' || rec.START_TIME::VARCHAR || '\n' ||
            '- Roles: ' || COALESCE(rec.ROLE_NAMES::VARCHAR, 'N/A') || '\n' ||
            '- Query Tag: ' || COALESCE(rec.QUERY_TAG, 'N/A') || '\n' ||
            '- Warehouse ID: ' || COALESCE(rec.WAREHOUSE_ID::VARCHAR, 'N/A') || '\n\n' ||
            'Please investigate this query and take appropriate action.'
        );
    END FOR;

    RETURN TABLE(result);
END;
$$
COMMENT = 'TOOL: Detect and cancel runaway AI queries (Expires: 2026-05-07)';

In [ ]:
-- D2. Hourly task — adjust the 50 credit threshold as needed
CREATE OR REPLACE TASK MONITOR_RUNAWAY_AI_QUERIES
    WAREHOUSE = SFE_AI_SPEND_CONTROLS_WH
    SCHEDULE  = 'USING CRON 0 * * * * UTC'
AS
    CALL MONITOR_AND_CANCEL_RUNAWAY_QUERIES(50);

ALTER TASK MONITOR_RUNAWAY_AI_QUERIES RESUME;

SELECT 'Runaway query monitor is active (hourly, threshold: 50 credits).' AS status;

---
## E — Verification & Status Checks
Confirm all governance objects are active and operating correctly.

In [ ]:
-- E1. Check alert status
SHOW ALERTS LIKE 'AI_FUNCTIONS_MONTHLY_SPEND_ALERT' IN SCHEMA SNOWFLAKE_EXAMPLE.AI_SPEND_CONTROLS;

In [ ]:
-- E2. Check task status
SHOW TASKS IN SCHEMA SNOWFLAKE_EXAMPLE.AI_SPEND_CONTROLS;

In [ ]:
-- E3. Alert execution history (last 24 hours)
SELECT
    SCHEDULED_TIME,
    NAME,
    STATE,
    CONDITION_QUERY_TEXT,
    ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.ALERT_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -1, CURRENT_TIMESTAMP()),
    ALERT_NAME => 'AI_FUNCTIONS_MONTHLY_SPEND_ALERT'
))
ORDER BY SCHEDULED_TIME DESC;

In [ ]:
-- E4. Per-user access control status
SELECT
    USER_NAME,
    MONTHLY_CREDIT_LIMIT,
    IS_ACTIVE,
    GRANTED_AT,
    REVOKED_AT,
    REVOCATION_REASON
FROM AI_FUNCTIONS_ACCESS_CONTROL
ORDER BY USER_NAME;

In [ ]:
-- E5. Which months have had alerts sent?
SELECT
    ALERT_NAME,
    ALERT_MONTH,
    SENT_AT,
    CREDITS_AT_ALERT
FROM AI_FUNCTIONS_ALERT_STATE
ORDER BY ALERT_MONTH DESC;

---
## Best Practices

- **Start with monitoring:** Establish baseline usage patterns (Section A) before enabling automated controls.
- **Set conservative initial limits:** Begin with lower thresholds and adjust upward based on actual patterns.
- **Use query tags:** Encourage teams to use `QUERY_TAG` session parameters for cost attribution by project or team.
- **Review regularly:** Periodically review the access control table and adjust per-user limits for legitimate needs.
- **Test alerts:** Set the threshold to 0 to trigger immediately, then reset after confirming email delivery.
- **Account for latency:** The ACCOUNT_USAGE view has up to 60 minutes of latency — factor this into monitoring.
- **Long-running exemptions:** Create a dedicated role (e.g., `AI_FUNCTIONS_USER_LONG_RUNNING_ROLE`) and exclude it from runaway detection.

Source: [Managing Cortex AI Function costs with Account Usage](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)